In [5]:
import yfinance as yf
import pandas as pd
from tickers import tickers_atuais

In [6]:
tickers = tickers_atuais()

In [8]:
dfx = pd.read_csv('ohlcv5y.csv')
df_ohlcv = pd.DataFrame(dfx)

dfy = pd.read_csv('fund.csv')
df_fund = pd.DataFrame(dfy)




In [9]:
df_ohlcv.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2021-02-19,A,124.101983,124.575535,122.246391,122.613647,1263900.0
1,2021-02-22,A,121.724491,121.927452,118.960441,119.356689,1340100.0
2,2021-02-23,A,119.366378,119.366378,116.370381,118.312943,2125400.0
3,2021-02-24,A,118.158285,121.202606,117.916672,120.825684,1898400.0
4,2021-02-25,A,120.622759,121.048000,117.839382,118.003677,1430900.0


vamos criar entao as features de avaliacao estudando os modelos

ideias geradas para fazer

In [ ]:
# ============================================================
# PREPARAÇÃO DOS DADOS PARA OS MODELOS
# ============================================================

# 1. Ordenar por Ticker e Date (CRÍTICO para séries temporais)
df = df_historico.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# 2. Criar features por ticker (indicadores técnicos)
def add_features(group):
    # Retornos
    group['return_1d'] = group['Close'].pct_change()
    group['return_5d'] = group['Close'].pct_change(5)
    
    # Médias móveis
    group['sma_5'] = group['Close'].rolling(5).mean()
    group['sma_20'] = group['Close'].rolling(20).mean()
    
    # Volatilidade (desvio padrão de 20 dias)
    group['volatility'] = group['return_1d'].rolling(20).std()
    
    # Volume relativo
    group['volume_ratio'] = group['Volume'] / group['Volume'].rolling(20).mean()
    
    # RSI simplificado (14 períodos)
    delta = group['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    group['RSI'] = 100 - (100 / (1 + gain / loss))
    
    # MACD simplificado
    ema_12 = group['Close'].ewm(span=12).mean()
    ema_26 = group['Close'].ewm(span=26).mean()
    group['MACD'] = ema_12 - ema_26
    
    # Bollinger Bands
    group['BB_upper'] = group['sma_20'] + 2 * group['Close'].rolling(20).std()
    group['BB_lower'] = group['sma_20'] - 2 * group['Close'].rolling(20).std()
    group['BB_position'] = (group['Close'] - group['BB_lower']) / (group['BB_upper'] - group['BB_lower'])
    
    return group

df = df.groupby('Ticker', group_keys=False).apply(add_features)

# 3. Criar TARGET (o que queremos prever)
# Target: 1 se o preço SOBE no próximo dia, 0 se DESCE
df['target'] = (df.groupby('Ticker')['Close'].shift(-1) > df['Close']).astype(int)

# 4. Remover linhas com NaN (início de cada série por causa dos indicadores)
df = df.dropna()

print(f"Shape final: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")
df.head(10)

In [ ]:
# ============================================================
# VERIFICAR SE ESTÁ PRONTO PARA CADA MODELO
# ============================================================

# Features que vamos usar
feature_cols = ['return_1d', 'return_5d', 'sma_5', 'sma_20', 'volatility', 
                'volume_ratio', 'RSI', 'MACD', 'BB_position']

print("=" * 60)
print("XGBOOST - Pronto para usar!")
print("=" * 60)
X_xgb = df[feature_cols]
y_xgb = df['target']
print(f"X shape: {X_xgb.shape}  (amostras × features)")
print(f"y shape: {y_xgb.shape}  (targets)")
print(f"Distribuição target: {y_xgb.value_counts().to_dict()}")

print("\n" + "=" * 60)
print("LSTM - Precisa converter para 3D")
print("=" * 60)

import numpy as np
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 20  # usando 20 porque temos só 1 mês de dados de teste

X_lstm, y_lstm = [], []
scaler = StandardScaler()

for ticker in df['Ticker'].unique():
    ticker_df = df[df['Ticker'] == ticker].sort_values('Date')
    
    # Normalizar features (CRÍTICO para LSTM)
    features_scaled = scaler.fit_transform(ticker_df[feature_cols])
    targets = ticker_df['target'].values
    
    # Criar sequências
    for i in range(SEQ_LEN, len(features_scaled)):
        X_lstm.append(features_scaled[i-SEQ_LEN:i])  # últimos N dias
        y_lstm.append(targets[i])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

print(f"X shape: {X_lstm.shape}  (amostras × sequência × features)")
print(f"y shape: {y_lstm.shape}  (targets)")

print("\n" + "=" * 60)
print("GARCH - Usa só retornos de cada ticker")
print("=" * 60)
print("Para GARCH, filtrar por ticker e usar coluna 'return_1d'")
print(f"Exemplo NVDA: {len(df[df['Ticker'] == df['Ticker'].iloc[0]])} observações")